In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import os

# ====================== UTILS (same as faces) ======================
def get_transforms(resize=224):
    return transforms.Compose([
        transforms.Resize((resize, resize)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

def get_dataloaders(data_dir, batch_size=32, num_workers=0):
    train_dir = os.path.join(data_dir, 'train')
    val_dir   = os.path.join(data_dir, 'val')
    test_dir  = os.path.join(data_dir, 'test')

    train_dataset = datasets.ImageFolder(train_dir, transform=get_transforms())
    val_dataset   = datasets.ImageFolder(val_dir,   transform=get_transforms())
    test_dataset  = datasets.ImageFolder(test_dir,  transform=get_transforms())

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=num_workers)

    class_names = train_dataset.classes
    return train_loader, val_loader, test_loader, class_names


# ====================== FINE-TUNE (same function) ======================
def fine_tune_model(data_dir, num_epochs=8, lr=0.001):
    os.makedirs("models", exist_ok=True)
    os.makedirs("plots", exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = models.resnet50(weights='IMAGENET1K_V1')
    num_ftrs = model.fc.in_features
    num_classes = len(datasets.ImageFolder(os.path.join(data_dir, 'train')).classes)
    model.fc = nn.Linear(num_ftrs, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD([
        {'params': model.fc.parameters(), 'lr': lr},
        {'params': model.layer4.parameters(), 'lr': lr / 10},
        {'params': model.layer3.parameters(), 'lr': lr / 100},
    ], momentum=0.9)

    train_loader, _, _, _ = get_dataloaders(data_dir, batch_size=32)

    print(f"Starting fine-tuning on {data_dir} | Classes: {num_classes} | Device: {device}")

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

    save_path = f"models/resnet50_finetuned_{os.path.basename(data_dir)}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"✅ Model saved: {save_path}")
    return model


# ====================== EXTRACT FEATURES (same) ======================
def extract_features(model, dataloader, device):
    model.eval()
    features = []
    labels = []
    
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Extracting features"):
            images = images.to(device)
            
            # Penultimate layer (2048-dim)
            x = model.conv1(images)
            x = model.bn1(x)
            x = model.relu(x)
            x = model.maxpool(x)
            x = model.layer1(x)
            x = model.layer2(x)
            x = model.layer3(x)
            x = model.layer4(x)
            x = model.avgpool(x)
            x = torch.flatten(x, 1)
            
            features.append(x.cpu().numpy())
            labels.append(targets.numpy())
    
    return np.vstack(features), np.concatenate(labels)


# ====================== VISUALIZATION (same) ======================
def plot_2d(X, y, title, class_names, method="PCA"):
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', s=60, alpha=0.8, edgecolors='k', linewidth=0.5)
    plt.title(f"{method} - {title}", fontsize=14)
    plt.xlabel(f"{method} 1")
    plt.ylabel(f"{method} 2")
    plt.colorbar(scatter, ticks=range(len(class_names)), label="Class")
    
    legend = plt.legend(*scatter.legend_elements(), title="Classes", fontsize=9)
    plt.setp(legend.get_texts(), fontsize='small')
    
    os.makedirs("plots", exist_ok=True)
    filename = f"plots/flowers_{title.replace(' ', '_').lower()}_{method.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")


# ====================== STEP 1: Prepare CIFAR-10 as train/val ======================
# This will download CIFAR-10 and split into train/val folders (you need to run once)
from torchvision import datasets as tv_datasets

cifar_root = "./data/cifar10"
try:
    cifar_train = tv_datasets.CIFAR10(root=cifar_root, train=True, download=True)
    cifar_test = tv_datasets.CIFAR10(root=cifar_root, train=False, download=True)
except Exception as e:
    print(f"Error downloading CIFAR-10: {e}")
    print("Please check your internet connection or download CIFAR-10 manually to './data/cifar10'.")
    raise

# Create ImageFolder structure for flowers (using CIFAR-10 classes)
import shutil
from pathlib import Path

flowers_dir = "data/flowers"
os.makedirs(f"{flowers_dir}/train", exist_ok=True)
os.makedirs(f"{flowers_dir}/val", exist_ok=True)

class_names_cifar = cifar_train.classes  # ['airplane', 'automobile', ..., 'truck']

for i, (img, label) in enumerate(cifar_train):
    class_name = class_names_cifar[label]
    dest = Path(f"{flowers_dir}/train/{class_name}")
    dest.mkdir(parents=True, exist_ok=True)
    img.save(dest / f"{i}.png")

# Use a portion of test set as val (or split properly)
val_split = int(0.2 * len(cifar_test))
for i, (img, label) in enumerate(cifar_test):
    class_name = class_names_cifar[label]
    if i < val_split:
        dest = Path(f"{flowers_dir}/val/{class_name}")
        dest.mkdir(parents=True, exist_ok=True)
        img.save(dest / f"{i}.png")
    else:
        # You can add the rest to train if you want more data
        pass

print("CIFAR-10 prepared as train/val in data/flowers/")


# ====================== STEP 2: Fine-tune on Flowers (CIFAR-10) ======================
print("\n=== Fine-tuning on Flowers (CIFAR-10) ===")
fine_tune_model("data/flowers", num_epochs=8)


# ====================== STEP 3: Extract Features Before & After ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# BEFORE
model_pre = models.resnet50(weights='IMAGENET1K_V1').to(device)
_, _, test_loader, class_names = get_dataloaders("data/flowers", batch_size=32)

X_test_pre, y_test = extract_features(model_pre, test_loader, device)

# AFTER
model_post = models.resnet50(weights=None)
num_classes = len(class_names)
model_post.fc = nn.Linear(model_post.fc.in_features, num_classes)
model_post.load_state_dict(torch.load("models/resnet50_finetuned_flowers.pth", 
                                      map_location=device, weights_only=True))
model_post = model_post.to(device)

X_test_post, _ = extract_features(model_post, test_loader, device)

# Save
os.makedirs("features", exist_ok=True)
np.save("features/flowers_pre_test.npy", X_test_pre)
np.save("features/flowers_post_test.npy", X_test_post)
np.save("features/flowers_test_labels.npy", y_test)

print("✅ Flowers features extracted and saved!")


# ====================== STEP 4: Visualization ======================
print("\n=== Generating Visualizations for Flowers ===")

X_pre  = np.load("features/flowers_pre_test.npy")
X_post = np.load("features/flowers_post_test.npy")
y      = np.load("features/flowers_test_labels.npy")

# Update with your actual flower class names (from test folder names)
class_names = ["rose", "sunflower", "tulip", "daisy"]   # ← CHANGE TO YOUR REAL FLOWER NAMES!

print(f"Classes: {class_names}")

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca_pre  = pca.fit_transform(X_pre)
X_pca_post = pca.fit_transform(X_post)
plot_2d(X_pca_pre,  y, "Before Fine-tuning", class_names, "PCA")
plot_2d(X_pca_post, y, "After Fine-tuning",  class_names, "PCA")

# t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne_pre  = tsne.fit_transform(X_pre)
X_tsne_post = tsne.fit_transform(X_post)
plot_2d(X_tsne_pre,  y, "Before Fine-tuning", class_names, "t-SNE")
plot_2d(X_tsne_post, y, "After Fine-tuning",  class_names, "t-SNE")

# UMAP
print("Running UMAP...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap_pre  = reducer.fit_transform(X_pre)
X_umap_post = reducer.fit_transform(X_post)
plot_2d(X_umap_pre,  y, "Before Fine-tuning", class_names, "UMAP")
plot_2d(X_umap_post, y, "After Fine-tuning",  class_names, "UMAP")

print("\n🎉 Flowers task completed! Check 'plots/' folder for flower plots.")